<a href="https://colab.research.google.com/github/mariobet23440/Proyecto-3---M-todos-Num-ricos-1/blob/main/Proyecto3MetodosNumericos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***MOTOR DC - GEMELO DIGITAL Y CONTROL PID***

```Proyecto 3 - Métodos Numéricos 1```

***Creado por:***
*   Mario Betancourt (23440)
*   Pongan sus nombres y (carnets)

## ***HISTORIAL DE REVISIONES***
*   21/11/2026 - Creación de archivo y método de Euler
*   22/11/2026 - Correcciones al desarrollo matemático de la Parte 1, revisadas y aprobadas por el equipo (ver `UsoIA/`)
*   22/11/2026 - Referencia a Nise (7.ª ed.) en la Introducción y sección de Referencias
*   22/11/2026 - Sección 1.4 reescrita (papel del Jacobiano en la estabilidad, regularización con tanh) y nueva sección 1.5 (espacios métricos y condición de Lipschitz), redactadas con IA



# ***INTRODUCCIÓN***
Le damos la bienvenida al cuaderno de Jupyter del Proyecto 3 del curso Métodos Numéricos 1 de la Universidad del Valle de Guatemala. En este proyecto trabajaremos en la aplicación de métodos numéricos para calcular derivadas e integrales y métodos de resolución de ecuaciones diferenciales al análisis de la dinámica de un motor DC.

De acuerdo con la guía, el motor DC se modela a través de un sistema de tres ecuaciones diferenciales acopladas:

$$\begin{cases}
L\frac{di}{dt} = V-Ri-K_e \omega \\
J_m\frac{d\omega}{dt} = K_t i - b\omega -\tau_c \text{ sign}(\omega) - \tau_{carga} \\
\frac{d\theta}{dt} = \omega
\end{cases}$$

Donde los símbolos representan las siguientes cantidades físicas

*   $L$ - Inductancia del motor [henrios, H]
*   $V$ - Voltaje aplicado en las terminales del motor [Volts, V]
*   $R$ - Resistencia del motor [Ohms, $\Omega$]
*   $K_e$ - Constante de fuerza contraelectromotriz [$V/(rad/s)$]
*   $\omega$ - Velocidad angular del motor [radianes por segundo, $rad/s$]
*   $J_m$ - Inercia del eje del motor [$\text{kg}\cdot\text{m}^2$]
*   $K_t$ - Constante de torque del motor [$\text{N}\cdot\text{m}/\text{A}$]
*   $b$ - Coeficiente de amortiguamiento viscoso [$\text{N}\cdot\text{m}\cdot\text{s}/\text{rad}$]
*   $\tau_c$ - Torque de fricción de Coulomb [$\text{N}\cdot\text{m}$]
*   $\tau_{carga}$ - Torque de carga [$\text{N}\cdot\text{m}$]

Un estudio de estos modelos en cursos de sistemas de control sugiere que el sistema de ecuaciones representa a un motor DC controlado por armadura. Este modelo de motores DC supone que la corriente en el estator (corriente de campo) es constante y el control de movimiento se realiza por la corriente del devanado del rotor (devanado de armadura). Este modelo de motores es aplicable también a motores DC de imanes permanentes. El motor DC JGA25-370 entra dentro de esta categoría. Para más información sobre este modelo en su versión lineal, véase Nise [1, cap. 2, §2.8, pp. 77–83].


A diferencia del modelo lineal presentado en Nise [1, §2.8], el modelo analizado en este proyecto agrega una dificultad adicional: *se considera un término no lineal para la fricción de Coulomb*. Aunque es posible linealizar el sistema y analizarlo a través de Transformadas de Laplace (como se hace en Sistemas de Control 1), ese no es el objetivo de este proyecto. Al fin y al cabo, el objetivo de este curso no es simplificar el modelo, sino resolverlo numéricamente tal cual. Por lo tanto, a diferencia de lo que se haría en un análisis de sistemas lineales, en este proyecto resolveremos la ecuación original con todas sus no linealidades.

# ***PARTE 1 - SOLUCIÓN DE SISTEMAS DE ECUACIONES DIFERENCIALES***

Ahora que fijamos como objetivo resolver el sistema de ecuaciones diferenciales mostrado anteriormente, surge la importante pregunta de cómo debemos proceder. Empecemos tratando de entender cómo resolver un sistema de ecuaciones a través del método de Euler.

## **1.1. El Método de Euler para una EDO de primer orden**
El método de Euler es fácil y muy intuitivo de definir para un Problema de Valores Iniciales (PVI) con una EDO de primer orden:

$$\begin{cases}
\frac{dy}{dx}=f(x,y)\\
y(x_0) = y_0
\end{cases}
$$

Donde $x_0$ y $y_0$ son constantes arbitrarias. Para aproximar la solución a este PVI, utilizamos diferencias finitas hacia adelante para aproximar la derivada en $x_k$ usando el intervalo $[x_k, x_{k+1}]$, donde $x_{k+1} = x_k + h$:

$$\frac{dy}{dx} \approx \frac{y_{k+1} - y_k}{h} $$

Donde $y_{k+1} \approx y(x_{k+1})$ y $y_{k} \approx y(x_{k})$ son las **aproximaciones numéricas** de la variable dependiente en los extremos del intervalo antes definido. (En la sección 1.4 se usará una letra mayúscula para distinguir la solución exacta.)

Al sustituir en la EDO, evaluando $f$ en el extremo izquierdo del intervalo, $(x_k, y_k)$:
$$\frac{y_{k+1} - y_k}{h}\approx f(x_k,y_k)$$

Evaluar $f$ en el extremo izquierdo es justamente lo que hace que el método sea **explícito**; evaluarla en $(x_{k+1}, y_{k+1})$ daría el método de Euler implícito.

Resuelva para $y_{k+1}$ y obtendrá la ecuación del método de Euler explícito (el método de Euler que aproxima la solución extrapolando hacia adelante usando la derivada en el punto actual):

$$y_{k+1} = y_k + hf(x_k, y_k)$$

¿Ve lo simple que es llegar a este famoso método?



## **1.2. El Método de Euler para un sistema de EDOs de primer orden**
Ahora extrapolemos esta deducción hacia un sistema de varias EDOs de variables dependientes en el vector $\mathbf{y}=[y_1, y_2, ..., y_N]^T$. ¿Qué tanto podría cambiar?

Suponga que todas las ecuaciones del sistema pueden escribirse en forma canónica (aunque el lado derecho sea una no linealidad espeluznante):

$$
\begin{cases}
y_1' = f_1(x,y_1,y_2, ..., y_N)\\
y_2' = f_2(x,y_1,y_2, ..., y_N)\\
y_3' = f_3(x,y_1,y_2, ..., y_N)\\
\vdots \\
y_N' = f_N(x,y_1,y_2, ..., y_N)
\end{cases}
$$

Aunque no lo crea, este sistema de ecuaciones grita que puede ser expresado en forma vectorial. Reúna las variables dependientes en el vector $\mathbf{y}$ y los lados derechos en una sola función vectorial $\mathbf{f}$, el **campo vectorial** del sistema:

$$
\mathbf{y} = \begin{pmatrix} y_1 \\ y_2 \\ \vdots \\ y_N \end{pmatrix},\qquad
\mathbf{f}(x,\mathbf{y}) = \begin{pmatrix} f_1(x,\mathbf{y}) \\ f_2(x,\mathbf{y}) \\ \vdots \\ f_N(x,\mathbf{y}) \end{pmatrix}
$$

Si recuerda que derivar una función vectorial equivale a derivar cada una de sus componentes, $\mathbf{y}' = [y_1', y_2', ..., y_N']^T$, todo el sistema cabe en una sola línea:

$$\mathbf{y}' = \mathbf{f}(x,\mathbf{y})$$

¡Para sistemas de EDOs de primer orden, la función de pendiente escalar $f(x,y)$ se reemplaza por el campo vectorial $\mathbf{f}(x,\mathbf{y})$!

Resolver el sistema de ecuaciones mediante el método de Euler es realmente sencillo: aproxime la derivada de cada variable dependiente en el sistema a través de diferencias finitas y resuelva para el valor de la función en el extremo derecho del intervalo sobre el que se aproxima. Básicamente, haga lo mismo que hicimos antes para deducir que $y_{k+1} = y_k + hf(x_k, y_k)$ pero *por cada función $y_i$ en el sistema de EDOs*.

De esta forma, por cada iteración $k$, $x_k = x_0 + hk$ y por cada variable dependiente:

$$
\begin{aligned}
(y_1)_{k+1} &= (y_1)_{k} + h f_1 (x_k, (y_1)_k, (y_2)_k, ..., (y_N)_k)\\
(y_2)_{k+1} &= (y_2)_{k} + h f_2 (x_k, (y_1)_k, (y_2)_k, ..., (y_N)_k)\\
&\ \ \vdots\\
(y_N)_{k+1} &= (y_N)_{k} + h f_N (x_k, (y_1)_k, (y_2)_k, ..., (y_N)_k)
\end{aligned}
$$

Estas $N$ fórmulas son exactamente las componentes de una sola ecuación vectorial. Por lo tanto, para un sistema de ecuaciones la fórmula iterativa del método de Euler puede reescribirse en forma vectorial:

$$\mathbf{y}_{k+1} =\mathbf{y}_{k} + h\,\mathbf{f}(x_k, \mathbf{y}_{k})$$

Esta forma de expresar la fórmula iterativa del método de Euler es extremadamente importante. Por favor, recuérdela, que será de mucha utilidad cuando tratemos de estudiar la estabilidad del método en este caso.

## **1.3. Sistemas de EDOs de orden superior**
Ahora considere que agregamos la complejidad adicional de que en nuestro sistema de ecuaciones tenemos al menos una ecuación con una derivada de orden superior de una variable dependiente. Suponga, por ejemplo, que en un sistema de ecuaciones que involucra las variables $\textbf{y} = [y_1, y_2, ..., y_N]^T$ la variable $y_i$, con $i \in \{1, 2, \dots, N\}$, tiene una derivada de orden $j$-ésimo en una ecuación del sistema.

La solución, en este caso, es bastante simple: *reemplace la derivada de orden superior por una derivada de primer orden utilizando variables auxiliares*.

Para ilustrar mejor esta idea, olvide que tratamos con sistemas de ecuaciones y considere lo que ocurre con una sola EDO de segundo orden:

$$\frac{d^2y}{dt^2} = \frac{dy}{dt} - 1$$

En este caso, podemos convertir la ecuación de segundo orden en un *sistema* de ecuaciones de primer orden sustituyendo la derivada de segundo orden por la derivada de una variable auxiliar. Considere que $u_1 = y$ y $u_2 = y'$. Para este problema se obtiene el sistema de EDOs de primer orden:

$$\begin{cases}
\frac{du_2}{dt}= u_2 - 1\\
\frac{du_1}{dt}= u_2
\end{cases}
$$

Por lo tanto, en un sistema de ecuaciones donde aparecen múltiples EDOs de orden superior, cada EDO de orden $n$ se descompone en exactamente $n$ EDOs de primer orden, usando las variables auxiliares $u_1 = y,\ u_2 = y',\ \dots,\ u_n = y^{(n-1)}$. En este ejemplo, dado que la ecuación era de segundo orden (orden 2), se necesitaron 2 ecuaciones de primer orden.

Observe que el modelo del motor ya está escrito como un sistema de primer orden, con vector de estado $\mathbf{y} = [i,\ \omega,\ \theta]^T$: la ecuación $\frac{d\theta}{dt} = \omega$ es justamente la reducción de orden de la ecuación de segundo orden para $\theta$.

Con esto mencionado, ya tenemos la teoría suficiente para (1) aplicar el método de Euler y generalizar a otros métodos para resolver EDOs vistos en el curso y (2) Analizar la estabilidad de la solución numérica, al menos para el método de Euler.

## **1.4. Estabilidad Numérica del Método de Euler**
Suponga que la solución exacta del sistema de EDOs en $x_k$ es $\mathbf{Y}(x_k)$ (observe que se usó una letra mayúscula). Esta puede calcularse de forma exacta en las cercanías de $x_k$ al utilizar series de Taylor y agregar un término de error local $\mathbf{T}_k$ por los términos de orden superior:

$$\mathbf{Y}(x_{k+1}) = \mathbf{Y}(x_k) + h\,\mathbf{f}(x_k, \mathbf{Y}(x_k)) + \mathbf{T}_k$$

Como $\mathbf{Y}$ es una función vectorial, el residuo de Taylor se escribe componente a componente: cada componente tiene su propio punto intermedio $\epsilon_i \in (x_k, x_{k+1})$,

$$(\mathbf{T}_k)_i = \frac{h^2}{2}\,Y_i''(\epsilon_i), \qquad i = 1, 2, \dots, N$$

En general no existe un único $\epsilon$ que sirva para todas las componentes a la vez. Lo que sí se puede afirmar es la cota

$$\|\mathbf{T}_k\| \le \frac{h^2}{2}\max_{x\in[x_k,x_{k+1}]}\|\mathbf{Y}''(x)\|$$

Definimos el error en el paso $k$ como $\mathbf{e}_{k} = \mathbf{Y}(x_{k}) - \mathbf{y}_{k}$. Al restar la fórmula de Euler $\mathbf{y}_{k+1} = \mathbf{y}_k + h\,\mathbf{f}(x_k, \mathbf{y}_k)$ de la expansión de Taylor, el error en el paso $k+1$ es:

$$\mathbf{e}_{k+1} = \mathbf{Y}(x_{k+1}) - \mathbf{y}_{k+1} = [\mathbf{Y}(x_k)-\mathbf{y}_{k}] + h[\mathbf{f}(x_k, \mathbf{Y}(x_k)) - \mathbf{f}(x_k, \mathbf{y}_k)] + \mathbf{T}_k$$

Este error tiene tres componentes:
- $[\mathbf{Y}(x_k) - \mathbf{y}_k] = \mathbf{e}_k$ es el error global acumulado hasta el paso $k$.
- $h[\mathbf{f}(x_k, \mathbf{Y}(x_k)) - \mathbf{f}(x_k, \mathbf{y}_k)]$ es el error propagado por la pendiente: muestra cómo el error en el paso anterior influye en el error de la pendiente del paso siguiente.
- $\mathbf{T}_k$ es el error de truncamiento local, se agrega por la Serie de Taylor.

Un método de resolución de sistemas de EDOs es **numéricamente estable** si, conforme $k$ aumenta, los errores que ya existen no crecen sin control: la solución numérica no "explota" (en pocas palabras). No hay que confundir esto con la **convergencia**, que pide que la solución numérica se acerque a la analítica cuando $h \to 0$. Las dos ideas se relacionan: un método *consistente* (cuyo error local cumple $\mathbf{T}_k/h \to \mathbf{0}$) y estable es convergente.

La definición de $\mathbf{e}_{k+1}$ sugiere que el error en el paso $k+1$ depende del error en el paso $k$ a través de la pendiente. Para saber si ese error crece o se atenúa hay que responder una pregunta: **¿cuánto cambia $\mathbf{f}$ cuando su argumento pasa de $\mathbf{y}_k$ a $\mathbf{Y}(x_k)$?** Esa es justamente la pregunta que responde la derivada de $\mathbf{f}$.

### **El Jacobiano: la derivada del campo vectorial**

Observe la diferencia de papeles. El método de Euler se calcula siempre con el campo vectorial $\mathbf{f}$, sea el sistema lineal o no. El Jacobiano, que es la derivada de $\mathbf{f}$, no forma parte del método: aparece solo al analizar cómo se propaga el error.

Recuerde cómo se define. Sea $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m$ una función vectorial diferenciable definida por $\mathbf{f}(\mathbf{x}) = (f_1(\mathbf{x}), f_2(\mathbf{x}), \dots, f_m(\mathbf{x}))$, donde $\mathbf{x} = (x_1, x_2, \dots, x_n)$. La *matriz jacobiana* de $\mathbf{f}$ en $\mathbf{x}$, denotada por $\mathbf{J}_{\mathbf{f}}(\mathbf{x})$ o $\frac{\partial \mathbf{f}}{\partial \mathbf{x}}$, es la matriz de $m \times n$ de sus primeras derivadas parciales:

$$
\mathbf{J}_{\mathbf{f}}(\mathbf{x}) = \frac{\partial (f_1, \dots, f_m)}{\partial (x_1, \dots, x_n)} =
\begin{pmatrix}
\frac{\partial f_1}{\partial x_1} & \frac{\partial f_1}{\partial x_2} & \cdots & \frac{\partial f_1}{\partial x_n} \\
\frac{\partial f_2}{\partial x_1} & \frac{\partial f_2}{\partial x_2} & \cdots & \frac{\partial f_2}{\partial x_n} \\
\vdots & \vdots & \ddots & \vdots \\
\frac{\partial f_m}{\partial x_1} & \frac{\partial f_m}{\partial x_2} & \cdots & \frac{\partial f_m}{\partial x_n}
\end{pmatrix}
$$

En nuestro caso interesa el Jacobiano del campo vectorial respecto de las variables dependientes $\mathbf{y}$, que es una matriz de $N \times N$:

$$
\mathbf{J}_{\mathbf{f}}(x,\mathbf{y}) = \frac{\partial \mathbf{f}}{\partial \mathbf{y}} =
\begin{pmatrix}
\frac{\partial f_1}{\partial y_1} & \cdots & \frac{\partial f_1}{\partial y_N} \\
\vdots & \ddots & \vdots \\
\frac{\partial f_N}{\partial y_1} & \cdots & \frac{\partial f_N}{\partial y_N}
\end{pmatrix}
$$

Por el teorema del valor medio en su forma integral, si $\mathbf{f}$ es diferenciable respecto de $\mathbf{y}$:

$$\mathbf{f}(x_k,\mathbf{Y}(x_k)) - \mathbf{f}(x_k,\mathbf{y}_k) = \left[\int_0^1 \mathbf{J}_{\mathbf{f}}\big(x_k,\ \mathbf{y}_k + s\,\mathbf{e}_k\big)\,ds\right]\mathbf{e}_k \approx \mathbf{J}_k\,\mathbf{e}_k$$

donde $\mathbf{J}_k := \mathbf{J}_{\mathbf{f}}(x_k,\mathbf{y}_k)$ es el Jacobiano evaluado en el paso $k$. La aproximación es exacta si $\mathbf{f}$ es afín en $\mathbf{y}$ (lineal más una constante); en general, el error que se comete es $O(\|\mathbf{e}_k\|^2)$. Al sustituir en la ecuación del error:

$$\mathbf{e}_{k+1} \approx \underbrace{(\mathbf{I} + h\mathbf{J}_k)\,\mathbf{e}_k}_{\text{propagación del error}} + \underbrace{\mathbf{T}_k}_{\text{error local nuevo}}$$

donde $\mathbf{I}$ es la matriz identidad de $N \times N$. La matriz $(\mathbf{I} + h\mathbf{J}_k)$ es la **matriz de amplificación**: dice cuánto se amplifica o se atenúa, en un paso, el error que ya existía.

### **¿Por qué la estabilidad no depende de $\mathbf{T}_k$?**

$\mathbf{T}_k$ no se ignora: es el error nuevo que se inyecta en cada paso, de tamaño $O(h^2)$. Pero no decide si el método es estable. Para verlo, suponga por simplicidad que el Jacobiano es constante, $\mathbf{J}_k = \mathbf{J}$, y desenrolle la recurrencia:

$$\mathbf{e}_n = (\mathbf{I} + h\mathbf{J})^n\,\mathbf{e}_0 + \sum_{j=0}^{n-1}(\mathbf{I} + h\mathbf{J})^{n-1-j}\,\mathbf{T}_j$$

Cada error que entra (el inicial $\mathbf{e}_0$ y cada $\mathbf{T}_j$) queda multiplicado por potencias de la misma matriz de amplificación:
- Si $(\mathbf{I} + h\mathbf{J})$ no amplifica, los errores locales simplemente se acumulan: $n$ términos de tamaño $O(h^2)$, con $n = (x_n - x_0)/h$, dan un error global $O(h)$. El método es estable.
- Si $(\mathbf{I} + h\mathbf{J})$ amplifica, sus potencias crecen sin control y hacen explotar tanto $\mathbf{e}_0$ como cada $\mathbf{T}_j$. El método es inestable.

Por eso el análisis de estabilidad estudia solo la parte de propagación, $\mathbf{e}_{k+1} \approx (\mathbf{I} + h\mathbf{J}_k)\,\mathbf{e}_k$. Si $\mathbf{J}$ es diagonalizable, con valores propios $\lambda_1, \dots, \lambda_N$, se obtiene el resultado usual, el **criterio de estabilidad absoluta** del método de Euler:

$$|1 + h\lambda_i| \le 1 \qquad \text{para todo } i = 1, \dots, N$$

Cuando $\mathbf{J}_k$ cambia de un paso a otro (sistema no lineal), el error acumulado es un producto de matrices distintas, $(\mathbf{I}+h\mathbf{J}_{n-1})\cdots(\mathbf{I}+h\mathbf{J}_1)(\mathbf{I}+h\mathbf{J}_0)\,\mathbf{e}_0$, y el criterio se aplica de forma local, con los valores propios del Jacobiano en cada zona de operación.

### **El caso del motor: la fricción de Coulomb**

El análisis anterior supone que $\mathbf{f}$ es diferenciable. En el modelo del motor esto falla en $\omega = 0$, donde $\text{sign}(\omega)$ salta de $-1$ a $1$ y no tiene derivada. Una solución habitual es **regularizar** la fricción con una función suave:

$$\text{sign}(\omega) \approx \tanh\left(\frac{\omega}{\varepsilon}\right), \qquad \varepsilon > 0 \text{ pequeño}$$

Esto es un cambio en el modelo, y conviene declararlo junto con el valor de $\varepsilon$ que se use. Como $\tanh$ es diferenciable, el Jacobiano existe en todas partes. Con el vector de estado $\mathbf{y} = [i,\ \omega,\ \theta]^T$:

$$
\mathbf{J}_{\mathbf{f}}(\omega) =
\begin{pmatrix}
-\dfrac{R}{L} & -\dfrac{K_e}{L} & 0 \\[6pt]
\dfrac{K_t}{J_m} & -\dfrac{b_{ef}(\omega)}{J_m} & 0 \\[6pt]
0 & 1 & 0
\end{pmatrix},
\qquad
b_{ef}(\omega) = b + \frac{\tau_c}{\varepsilon}\,\text{sech}^2\left(\frac{\omega}{\varepsilon}\right)
$$

La fricción de Coulomb aparece como un amortiguamiento efectivo $b_{ef}(\omega)$ que depende de la velocidad:
- Lejos de cero ($|\omega| \gg \varepsilon$), $\text{sech}^2 \to 0$ y $b_{ef} \approx b$: el Jacobiano es la matriz constante del modelo sin fricción de Coulomb.
- Cerca de cero, $b_{ef} \approx b + \tau_c/\varepsilon$, un valor muy grande. Ese término vuelve al sistema **rígido** cerca de $\omega = 0$: obligará a usar pasos $h$ pequeños, tanto más pequeños cuanto menor sea $\varepsilon$.

El cálculo de los valores propios de $\mathbf{J}_{\mathbf{f}}(\omega)$ con los parámetros del motor, y el paso máximo que resulta del criterio de estabilidad absoluta, se desarrollará más adelante.

## **1.5. Bonus: estabilidad del método de Euler con la condición de Lipschitz**

El análisis de la sección 1.4 tiene dos limitaciones: necesita que $\mathbf{f}$ sea diferenciable, y es local, porque se apoya en la aproximación $\mathbf{f}(x_k,\mathbf{Y}(x_k)) - \mathbf{f}(x_k,\mathbf{y}_k) \approx \mathbf{J}_k\,\mathbf{e}_k$. En esta sección se generaliza el resultado con una herramienta que no necesita derivadas: la **condición de Lipschitz**. Para enunciarla hay que precisar antes qué significa "distancia".

### **Espacios métricos y funciones distancia**

Un **espacio métrico** es un par $(M, d)$ formado por un conjunto $M$ y una **función distancia** (o métrica) $d: M \times M \to \mathbb{R}$ que, para todos $a, b, c \in M$, cumple:

1. **No negatividad:** $d(a,b) \ge 0$, y $d(a,b) = 0$ si y solo si $a = b$.
2. **Simetría:** $d(a,b) = d(b,a)$.
3. **Desigualdad triangular:** $d(a,c) \le d(a,b) + d(b,c)$.

La función distancia formaliza la idea intuitiva de "qué tan lejos" están dos elementos del conjunto. El ejemplo que nos interesa es $\mathbb{R}^N$ con la distancia inducida por una norma:

$$d(\mathbf{a},\mathbf{b}) = \|\mathbf{a} - \mathbf{b}\|$$

Cualquier norma sirve; por ejemplo, la euclidiana $\|\mathbf{v}\|_2 = \sqrt{v_1^2 + \dots + v_N^2}$ o la del máximo $\|\mathbf{v}\|_\infty = \max_i |v_i|$. Con esta notación, la norma del error es una distancia: $\|\mathbf{e}_k\| = d\big(\mathbf{Y}(x_k), \mathbf{y}_k\big)$ mide qué tan lejos está la solución numérica de la exacta en el paso $k$.

### **Funciones Lipschitz**

Sean $(X, d_X)$ y $(Z, d_Z)$ dos espacios métricos. Una función $g: X \to Z$ es **Lipschitz** si existe una constante $\mathcal{L} \ge 0$ tal que

$$d_Z\big(g(a), g(b)\big) \le \mathcal{L}\, d_X(a, b) \qquad \text{para todos } a, b \in X$$

La menor constante que cumple esta desigualdad se llama **constante de Lipschitz**. (Se usa $\mathcal{L}$ para no confundirla con la inductancia $L$ del motor.) La interpretación es directa: una función Lipschitz no "estira" las distancias más que un factor $\mathcal{L}$. Dos puntos cercanos siguen cercanos después de aplicar $g$: en un solo paso, la distancia entre las imágenes no puede crecer más de $\mathcal{L}$ veces, así que no puede explotar.

Para un sistema de EDOs se usa esta condición respecto de las variables dependientes: $\mathbf{f}$ es Lipschitz en $\mathbf{y}$ si existe $\mathcal{L}$ tal que

$$\|\mathbf{f}(x,\mathbf{Y}) - \mathbf{f}(x,\mathbf{y})\| \le \mathcal{L}\,\|\mathbf{Y} - \mathbf{y}\| \qquad \text{para todo } x \text{ del intervalo y todos } \mathbf{Y}, \mathbf{y}$$

Esta condición tiene una consecuencia importante que no se demuestra aquí: garantiza que el PVI tenga solución y que esa solución sea única (teorema de Picard-Lindelöf).

**Relación con el Jacobiano.** Si $\mathbf{f}$ es diferenciable respecto de $\mathbf{y}$ en una región convexa y su Jacobiano está acotado ahí, entonces $\mathbf{f}$ es Lipschitz en esa región con

$$\mathcal{L} = \sup \|\mathbf{J}_{\mathbf{f}}\|$$

donde la norma de la matriz es la inducida por la norma vectorial elegida. Lo contrario no es cierto: una función puede ser Lipschitz sin ser diferenciable. Por ejemplo, $g(x) = |x|$ es Lipschitz con $\mathcal{L} = 1$, aunque no tiene derivada en $x = 0$.

**Ejemplos del motor:**
- $\tanh(\omega/\varepsilon)$ es Lipschitz con $\mathcal{L} = 1/\varepsilon$, porque su derivada cumple $\frac{1}{\varepsilon}\,\text{sech}^2(\omega/\varepsilon) \le \frac{1}{\varepsilon}$.
- $\text{sign}(\omega)$ **no** es Lipschitz: ni siquiera es continua en $\omega = 0$. Dos velocidades $\omega = -\delta$ y $\omega = \delta$, tan cercanas como se quiera, tienen imágenes a distancia $2$.

### **Cota del error global del método de Euler**

Suponga que $\mathbf{f}$ es Lipschitz en $\mathbf{y}$ con constante $\mathcal{L}$, y que $\|\mathbf{Y}''(x)\| \le M$ en todo el intervalo, de modo que $\|\mathbf{T}_k\| \le \frac{h^2}{2}M$ (sección 1.4). Al tomar normas en la ecuación del error y usar la desigualdad triangular:

$$\|\mathbf{e}_{k+1}\| \le \|\mathbf{e}_k\| + h\,\|\mathbf{f}(x_k,\mathbf{Y}(x_k)) - \mathbf{f}(x_k,\mathbf{y}_k)\| + \|\mathbf{T}_k\| \le (1 + h\mathcal{L})\,\|\mathbf{e}_k\| + \frac{h^2}{2}M$$

Observe que aquí no se aproximó nada ni se derivó: la condición de Lipschitz reemplaza al Jacobiano. Al aplicar la desigualdad repetidamente, desde $k = 0$ hasta $k = n-1$:

$$\|\mathbf{e}_n\| \le (1+h\mathcal{L})^n\,\|\mathbf{e}_0\| + \frac{h^2 M}{2}\sum_{j=0}^{n-1}(1+h\mathcal{L})^j$$

La suma es geométrica:

$$\sum_{j=0}^{n-1}(1+h\mathcal{L})^j = \frac{(1+h\mathcal{L})^n - 1}{h\mathcal{L}}$$

Por último, como $1 + z \le e^{z}$ para todo $z$, se cumple $(1+h\mathcal{L})^n \le e^{nh\mathcal{L}} = e^{\mathcal{L}(x_n - x_0)}$, ya que $nh = x_n - x_0$. Al juntar todo:

$$\boxed{\|\mathbf{e}_n\| \le e^{\mathcal{L}(x_n - x_0)}\,\|\mathbf{e}_0\| + \frac{hM}{2\mathcal{L}}\left(e^{\mathcal{L}(x_n - x_0)} - 1\right)}$$

Esta cota dice dos cosas:
1. **Estabilidad:** un error inicial $\mathbf{e}_0$ (por ejemplo, de redondeo) se amplifica como mucho por el factor $e^{\mathcal{L}(x_n - x_0)}$, que no depende de $h$: queda acotado en cualquier intervalo finito.
2. **Convergencia de orden 1:** si $\mathbf{e}_0 = \mathbf{0}$, el error global es $O(h)$: al reducir $h$ a la mitad, la cota del error se reduce a la mitad. Este término viene de $\mathbf{T}_k$: el error local no se ignoró, sino que se fue acumulando.

### **Alcances y limitaciones**

- **Ventaja:** el resultado vale para sistemas no lineales en todo el intervalo, sin linealizar y sin necesitar derivadas.
- **Limitación 1: la cota es muy pesimista.** El factor $e^{\mathcal{L}(x_n - x_0)}$ crece muy rápido con $\mathcal{L}$. En el motor con la regularización $\tanh$, la constante de Lipschitz es al menos del orden de $\max\{R/L,\ \tau_c/(J_m\varepsilon)\}$, que puede ser de miles de $\text{s}^{-1}$; en un intervalo de un segundo, el factor sería astronómico. La cota garantiza que el método converge cuando $h \to 0$, pero no sirve para elegir $h$ en la práctica. Para eso se usa el criterio de estabilidad absoluta de la sección 1.4.
- **Limitación 2: requiere que $\mathbf{f}$ sea Lipschitz.** Con $\text{sign}(\omega)$, el teorema no se aplica cuando $\omega$ cruza cero. La regularización con $\tanh$ lo resuelve, a cambio de una constante $\mathcal{L}$ grande.

# ***REFERENCIAS***

[1] N. S. Nise, *Control Systems Engineering*, 7th ed. Hoboken, NJ, USA: John Wiley & Sons, 2015.